In [1]:
!pip install pinecone

In [2]:
!pip install langchain langchain-core langchain-openai langchain-community \
            langchain-chroma chromadb pymupdf python-dotenv langchain-huggingface sentence-transformers

In [3]:
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model="qwen3-4b",
    temperature=0,
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\ahmed\Rag_project_software\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5358.04it/s]


# Chunking Strategies: Size and Overlap

## Recommended Chunk Sizes

| Size Range | Best For |
|------------|----------|
| 200–400 | Product catalogs, FAQs, short facts |
| 500–1000 | General documents, articles |
| 1000–2000 | Legal docs, technical manuals needing more context |

## Why Chunks Share Text: The Overlap Principle

### The Problem Without Overlap

When splitting text exactly at chunk boundaries without overlap, important context gets lost:



**Issues:**
- **Chunk 1:** Has the product name and design, but no size/availability info
- **Chunk 2:** Has size options, but no product context

### The Solution: Using `chunk_overlap=50`

By overlapping 50 characters between chunks, we preserve essential context:



**Benefits:**
- **Chunk 1:** Product name + design ✓
- **Chunk 2:** Design (context) + availability info ✓

Both chunks are now independently understandable when retrieved, maintaining semantic coherence and reducing context loss at boundaries.

### Implementation

```python
chunk_overlap = 50

In [4]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyMuPDFLoader("C:\\Users\\ahmed\\Rag_project_software\\nike_football_catalog.pdf")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,       # small — see note below
    chunk_overlap=50,
    separators=["\n\n", "\n", " "],
)
chunks = splitter.split_documents(docs)
print(f"Total chunks: {len(chunks)}")

Total chunks: 178


# Understanding Cosine Similarity in Vector Retrieval

## What Is Cosine Similarity?

Cosine similarity measures **how similar two vectors are** by calculating the angle between them in high-dimensional space. It returns a score between **-1 and 1**, where:

- **1** = identical direction (most similar)
- **0** = perpendicular (no similarity)
- **-1** = opposite direction (least similar)

In RAG systems, we only care about positive values (0 to 1).

## How It Works in Your System

### Step 1: Chunks Become Vectors

When you load documents and chunk them, the embedding model converts each chunk into a **vector** (list of numbers):

```
Chunk 1: "Arsenal away kit features striking blue design"
↓ (MiniLM embedding)
Vector 1: [0.23, -0.15, 0.88, 0.41, -0.12, ...]  (384 dimensions)

Chunk 2: "It is available in sizes S–XXL"
↓ (MiniLM embedding)
Vector 2: [0.19, -0.12, 0.90, 0.38, -0.10, ...]  (384 dimensions)
```

### Step 2: Query Becomes a Vector

When you ask a question, it's also converted to a vector:

```
Query: "What sizes does the Arsenal kit come in?"
↓ (same MiniLM embedding)
Query Vector: [0.21, -0.13, 0.89, 0.39, -0.11, ...]
```

### Step 3: Cosine Similarity Comparison

The retriever calculates the angle between the query vector and **all chunk vectors**:

```
Similarity(Query, Chunk 1) = 0.95  ← Very similar (both about Arsenal kit)
Similarity(Query, Chunk 2) = 0.92  ← Similar (mentions sizes)
Similarity(Query, Chunk 3) = 0.45  ← Not very similar (about shoes)
Similarity(Query, Chunk 4) = 0.12  ← Very different (about pricing policy)
```

### Step 4: Return Top-K Results

With `search_kwargs={"k": 3}`, the retriever returns the **3 highest scores**:

```
1. Chunk 1 (0.95) → "Arsenal away kit features striking blue design"
2. Chunk 2 (0.92) → "It is available in sizes S–XXL"
3. Chunk 3 (0.45) → "Premium materials, durable construction..."
```

## Why Cosine Similarity Works

**Semantic meaning is preserved in vector space:**

- Vectors with similar **meaning** point in similar **directions**
- "Arsenal shirt sizes" and "Arsenal kit availability" have similar angles → high cosine similarity
- "Nike shoes pricing" and "Arsenal kit" point in different directions → low cosine similarity

## The Formula (Optional Deep Dive)

$$\cos(\theta) = \frac{\vec{A} \cdot \vec{B}}{|\vec{A}| \times |\vec{B}|}$$

Where:
- $\vec{A} \cdot \vec{B}$ = dot product (sum of element-wise multiplications)
- $|\vec{A}|$ and $|\vec{B}|$ = magnitudes (lengths) of vectors

**Example with 2D vectors:**
```
Vector A: [1, 0]
Vector B: [1, 1]

Dot product: (1×1) + (0×1) = 1
Magnitude of A: √(1² + 0²) = 1
Magnitude of B: √(1² + 1²) = √2

Cosine Similarity = 1 / (1 × √2) ≈ 0.707
```

## Why This Matters for Your RAG System

| Scenario | Cosine Similarity | Result |
|----------|-------------------|--------|
| User asks: "Arsenal shirts" | Query vs Chunk about "Arsenal kit": 0.94 | ✓ Retrieved |
| User asks: "Arsenal shirts" | Query vs Chunk about "shoes": 0.25 | ✗ Not in top 3 |
| User asks: "Cheap products" | Query vs Chunk with prices: 0.88 | ✓ Retrieved |

**The embedding model learns semantic relationships**, so similar concepts get high cosine similarity scores even if the exact words differ.

In [5]:
#only once
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks, # Your 300-char text chunks
    embedding=embeddings, # The MiniLM model converts each chunk to a vector
    persist_directory="./nike_chroma_db", # Saves to disk so you don't re-embed every run
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [6]:
# what happens after the first run
vectorstore = Chroma(
    persist_directory="./nike_chroma_db",
    embedding_function=embeddings,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Understanding the @tool Decorator

## What @tool Does

The `@tool` decorator from LangChain automatically converts a Python function into a **structured tool** that the LLM can understand and use.

### Four Key Steps

1. **Extracts the docstring** → Becomes the tool's description
2. **Inspects function parameters** → Creates schema (e.g., `query` is a string)
3. **Wraps the function** → Creates a Tool object with name, description, and schema
4. **Makes it LLM-readable** → Converts to JSON schema

## The Tool Schema (What LLM Receives)

When you call `llm.bind_tools(tools)`, the LLM receives a structured JSON representation:

```json
{
  "name": "nike_search",
  "description": "Search the Nike product catalog. Use for questions about shirts, boots, shorts, club kits, prices, or availability.",
  "parameters": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "The search query"
      }
    },
    "required": ["query"]
  }
}
```

### Schema Breakdown

| Field | Meaning |
|-------|---------|
| `name` | Function name — how the LLM calls it |
| `description` | **From docstring** — tells LLM when to use this tool |
| `parameters` | Function signature in JSON schema format |
| `properties.query` | Parameter name and type |
| `required` | Which parameters must be provided |

## How the LLM Uses This

When a user asks a question, the LLM:

1. Reads the user query
2. Scans all available tool descriptions
3. Matches the query intent to a tool description
4. Decides whether to call a tool
5. Extracts the correct parameters from the user's message
6. Returns tool call with args (e.g., `{"query": "Arsenal shirts"}`)

### Example

**User:** "What Arsenal shirts do you have?"

**LLM's Thinking:**
- Question is about "shirts" ✓
- `nike_search` description mentions "shirts" ✓
- This is the right tool!
- Parameter needed: `query = "Arsenal shirts"`

**LLM Calls:** `nike_search(query="Arsenal shirts")`

## Why Your Docstring Matters

Your docstring is **critical** for tool selection:

```python
"""Search the Nike product catalog. Use for questions about shirts,
boots, shorts, club kits, prices, or availability."""
```

This tells the LLM exactly when to use this tool. A vague docstring leads to wrong tool choices. A clear one ensures accurate routing.

In [7]:
from langchain.tools import tool

@tool
def nike_search(query: str) -> str:
    """Search the Nike product catalog. Use for questions about shirts,
    boots, shorts, club kits, prices, or availability."""
    docs = retriever.invoke(query) # Semantic search → top 3 chunks
    return "\n\n".join(d.page_content for d in docs) # Joins chunks into one string for the LLM

@tool
def filter_by_price(query: str, max_price: float) -> str:
    """Search Nike products and return only those under a given price in GBP."""
    docs = retriever.invoke(query) # Retrieve relevant chunks
    results = []
    for d in docs:
        text = d.page_content
        import re
        prices = re.findall(r'£([\d.]+)', text)  # Extract all "£12.99"-style prices using regex
        if prices and all(float(p) <= max_price for p in prices): # Keep chunk only if ALL prices fit budget
            results.append(text)
    return "\n\n".join(results) if results else "No products found under that price."

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

def run_nike_agent(question: str):
    tools = [nike_search, filter_by_price]
    tools_dict = {t.name: t for t in tools}
    llm_with_tools = llm.bind_tools(tools)

    messages = [
        SystemMessage(content=(
            "You are a Nike UK football product assistant. "
            "Use the nike_search tool to find products. "
            "Use filter_by_price when the user mentions a budget. "
            "Always mention the price in GBP in your final answer."
        )),
        HumanMessage(content=question),
    ]

    for i in range(1, 10):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            print(f"\nAnswer: {response.content}")
            return response.content

        for tc in response.tool_calls:
            result = tools_dict[tc["name"]].invoke(tc["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

run_nike_agent("do you have barcelona merch")


Answer: 

Yes, we have a range of F.C. Barcelona merchandise available! Here are some of the items:

- **Kids' Nike Dri-FIT Football Short-Sleeve Top (Strike Fourth Older)** - £59.99
- **Men's Nike Football T-Shirt (Swoosh)** - £32.99
- **Men's Nike Dri-FIT ADV Football Authentic Shirt (2025/26 Match Fourth)** - £19.99
- **G FC Barcelona 2025/26 Stadium** - £124.99

Let me know if you'd like more details on any of these or if you have a budget in mind!


"\n\nYes, we have a range of F.C. Barcelona merchandise available! Here are some of the items:\n\n- **Kids' Nike Dri-FIT Football Short-Sleeve Top (Strike Fourth Older)** - £59.99\n- **Men's Nike Football T-Shirt (Swoosh)** - £32.99\n- **Men's Nike Dri-FIT ADV Football Authentic Shirt (2025/26 Match Fourth)** - £19.99\n- **G FC Barcelona 2025/26 Stadium** - £124.99\n\nLet me know if you'd like more details on any of these or if you have a budget in mind!"